## Dimension Table - MED834ENRL_OLAP Schema  

In [0]:
spark.sql('''
  CREATE TABLE IF NOT EXISTS DIM_GROUP (
    GRGR_KEY BIGINT NOT NULL,
    GRGR_ID VARCHAR(30) NOT NULL,
    GROUP_NUMBER VARCHAR(20) NOT NULL,
    GROUP_NAME VARCHAR(200) NOT NULL,
    STATE_CODE VARCHAR(2),
    EFFECTIVE_DATE DATE NOT NULL,
    END_DATE DATE,
    CURRENT_FLAG BOOLEAN NOT NULL
  )
  USING DELTA
  COMMENT 'Employer Group Dimension - SCD Type 2. Represents an organization or employer group associated with healthcare enrollment.'
''')

In [0]:
spark.sql('''
  CREATE TABLE IF NOT EXISTS DIM_MEMBER (
    MEME_KEY BIGINT NOT NULL,
    MEME_ID VARCHAR(30) NOT NULL,
    MESB_ID VARCHAR(30) NOT NULL,
    FIRST_NAME VARCHAR(100) NOT NULL,
    LAST_NAME VARCHAR(100) NOT NULL,
    DATE_OF_BIRTH DATE NOT NULL,
    GENDER_CODE VARCHAR(1),
    RELATIONSHIP_CODE VARCHAR(2),
    EFFECTIVE_DATE DATE NOT NULL,
    END_DATE DATE,
    CURRENT_FLAG BOOLEAN NOT NULL
  )
  USING DELTA
  COMMENT 'Member Dimension - SCD Type 2. Maintains historical versions of member attributes.'
''')

In [0]:
spark.sql('''
  CREATE TABLE IF NOT EXISTS DIM_PLAN (
    PLPL_KEY BIGINT NOT NULL,
    PLPL_ID VARCHAR(30) NOT NULL,
    PLAN_CODE VARCHAR(20) NOT NULL,
    PLAN_NAME VARCHAR(200) NOT NULL,
    LINE_OF_BUSINESS_CODE VARCHAR(30) NOT NULL,
    EFFECTIVE_DATE DATE NOT NULL,
    END_DATE DATE,
    CURRENT_FLAG BOOLEAN NOT NULL
  )
  USING DELTA
  COMMENT 'Benefit Plan Dimension - SCD Type 2. Maintains historical versions of benefit plan attributes.'
''')

In [0]:
spark.sql('''
  CREATE TABLE IF NOT EXISTS DIM_DATE (
    DATE_KEY INT NOT NULL,
    FULL_DATE DATE NOT NULL,
    YEAR_NUMBER INT NOT NULL,
    QUARTER_NUMBER INT NOT NULL,
    MONTH_NUMBER INT NOT NULL,
    DAY_NUMBER INT NOT NULL,
    DAY_OF_WEEK_NAME VARCHAR(20) NOT NULL
  )
  USING DELTA
  COMMENT 'Calendar Dimension. DATE_KEY uses YYYYMMDD format (e.g., 20260827 = 27-Aug-2026)'
''')

In [0]:
spark.sql('''
  CREATE TABLE IF NOT EXISTS DIM_TRANSACTION_SOURCE (
    TXN_SRC_KEY BIGINT NOT NULL,
    TXN_ID BIGINT NOT NULL,
    FILE_NAME VARCHAR(255) NOT NULL,
    SENDER_ID VARCHAR(20) NOT NULL,
    RECEIVED_TIMESTAMP TIMESTAMP NOT NULL
  )
  USING DELTA
  COMMENT 'Transaction Source Dimension. Provides analytical lineage back to the inbound 834 transaction.'
''')

In [0]:
spark.sql('''
  CREATE TABLE IF NOT EXISTS FACT_ENROLLMENT (
    ENROLLMENT_FACT_KEY BIGINT NOT NULL,
    MEME_KEY BIGINT NOT NULL,
    PLPL_KEY BIGINT NOT NULL,
    GRGR_KEY BIGINT NOT NULL,
    TXN_SRC_KEY BIGINT NOT NULL,
    COVERAGE_START_DATE_KEY INT NOT NULL,
    COVERAGE_END_DATE_KEY INT,
    MAINTENANCE_TYPE_CODE VARCHAR(3) NOT NULL,
    ENROLLMENT_STATUS VARCHAR(15) NOT NULL,
    ENROLLMENT_COUNT INT NOT NULL,
    LOAD_TIMESTAMP TIMESTAMP NOT NULL
  )
  USING DELTA
  COMMENT 'Enrollment Fact. Grain: one row per enrollment event. Measures enrollment count with dimensional context.'
''')

In [0]:
# Verify all tables were created successfully
print("Schema Tables Created:")
print("=" * 50)

tables = spark.sql("SHOW TABLES").filter("tableName IN ('dim_group', 'dim_member', 'dim_plan', 'dim_date', 'dim_transaction_source', 'fact_enrollment')")
display(tables)

print(f"\n✅ Total tables created: {tables.count()}")